In [1]:
# ============================================================
# CELL 1 — Load Dataset & Configuration
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data/processed/cpcb_era5_2021_2025.csv"

TARGETS = ["co","no2","o3","pm10","pm25","so2"]

ERA5_FEATURES = [
    "era5_t2m_c","era5_d2m_c","era5_sp_hpa",
    "era5_tp_mm","era5_wind_speed","era5_wind_dir","era5_blh"
]

df = pd.read_csv(DATA_PATH, parse_dates=["datetime_utc"])

print("="*60)
print("DATASET LOADED")
print("="*60)
print(f"Rows     : {len(df):,}")
print(f"Stations : {df.location_id.nunique()}")
print(f"Date     : {df.datetime_utc.min()} → {df.datetime_utc.max()}")
print(f"Columns  : {len(df.columns)}")

DATASET LOADED
Rows     : 1,621,414
Stations : 37
Date     : 2021-01-01 00:00:00 → 2025-12-31 21:00:00
Columns  : 21


In [2]:
# ============================================================
# CELL 2 — Preprocessing Validation
# ============================================================

print("="*60)
print("PREPROCESSING VALIDATION")
print("="*60)

print(
    "Duplicate station-times:",
    df.duplicated(["location_id","datetime_utc"]).sum()
)

print(
    "Missing coordinates:",
    df[["latitude","longitude"]].isna().any(axis=1).sum()
)

print(
    "Missing ERA5 values:",
    df[ERA5_FEATURES].isna().sum().sum()
)

print("\nTarget missingness (%):")
print((df[TARGETS].isna().mean()*100).round(2))

PREPROCESSING VALIDATION
Duplicate station-times: 0
Missing coordinates: 0
Missing ERA5 values: 0

Target missingness (%):
co      13.35
no2     10.85
o3      11.41
pm10    13.09
pm25    11.34
so2     21.95
dtype: float64


In [3]:
# ============================================================
# CELL 3 — Temporal Features
# ============================================================

dt = df["datetime_utc"]

df["hour"] = dt.dt.hour
df["month"] = dt.dt.month
df["dayofweek"] = dt.dt.dayofweek

df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
df["dow_sin"] = np.sin(2*np.pi*df["dayofweek"]/7)
df["dow_cos"] = np.cos(2*np.pi*df["dayofweek"]/7)

print("Time features added.")

Time features added.


In [4]:
# ============================================================
# CELL 4 — Historical Pollutant Features
# ============================================================

df = df.sort_values(
    ["location_id","datetime_utc"]
).reset_index(drop=True)

g = df.groupby("location_id", group_keys=False)

for col in TARGETS:
    df[f"{col}_lag1"] = g[col].shift(1)
    df[f"{col}_lag24"] = g[col].shift(24)
    df[f"{col}_roll24_mean"] = g[col].transform(
        lambda s: s.shift(1).rolling(24, min_periods=1).mean()
    )

hist_features = [
    f"{p}_{x}"
    for p in TARGETS
    for x in ["lag1","lag24","roll24_mean"]
]

print(f"Historical features added: {len(hist_features)}")
print(hist_features)

Historical features added: 18
['co_lag1', 'co_lag24', 'co_roll24_mean', 'no2_lag1', 'no2_lag24', 'no2_roll24_mean', 'o3_lag1', 'o3_lag24', 'o3_roll24_mean', 'pm10_lag1', 'pm10_lag24', 'pm10_roll24_mean', 'pm25_lag1', 'pm25_lag24', 'pm25_roll24_mean', 'so2_lag1', 'so2_lag24', 'so2_roll24_mean']


In [5]:
# ============================================================
# CELL 5 — Chronological ~70/15/15 Split
# ============================================================

times = np.sort(df["datetime_utc"].unique())
n = len(times)

t1 = times[int(n*0.70)]
t2 = times[int(n*0.85)]

train = df[df.datetime_utc < t1].copy()
val   = df[(df.datetime_utc >= t1) & (df.datetime_utc < t2)].copy()
test  = df[df.datetime_utc >= t2].copy()

print("="*60)
print("CHRONOLOGICAL ~70/15/15 SPLIT")
print("="*60)

for name, part in [
    ("Train", train),
    ("Val", val),
    ("Test", test)
]:
    print(f"{name}: {len(part):,} rows ({len(part)/len(df)*100:.2f}%)")
    print(
        f"      {part.datetime_utc.min()} → "
        f"{part.datetime_utc.max()}"
    )

CHRONOLOGICAL ~70/15/15 SPLIT
Train: 1,134,975 rows (70.00%)
      2021-01-01 00:00:00 → 2024-07-02 02:00:00
Val: 243,201 rows (15.00%)
      2024-07-02 03:00:00 → 2025-04-01 23:00:00
Test: 243,238 rows (15.00%)
      2025-04-02 00:00:00 → 2025-12-31 21:00:00


In [6]:
# ============================================================
# CELL 6 — Temporal Leakage Check
# ============================================================

checks = {
    "Train max < Val min":
        train.datetime_utc.max() < val.datetime_utc.min(),

    "Val max < Test min":
        val.datetime_utc.max() < test.datetime_utc.min(),

    "No train/val overlap":
        ~train.datetime_utc.isin(val.datetime_utc).any(),

    "No val/test overlap":
        ~val.datetime_utc.isin(test.datetime_utc).any()
}

print("="*60)
print("TEMPORAL LEAKAGE CHECK")
print("="*60)

for name, ok in checks.items():
    print(("PASS" if ok else "FAIL") + " — " + name)

if not all(checks.values()):
    raise ValueError("Temporal leakage detected.")

TEMPORAL LEAKAGE CHECK
PASS — Train max < Val min
PASS — Val max < Test min
PASS — No train/val overlap
PASS — No val/test overlap


In [7]:
# ============================================================
# CELL 7 — Feature Inventory & Missingness
# ============================================================

FEATURES = (
    ["latitude","longitude"] +
    ERA5_FEATURES +
    [
        "hour","month","dayofweek",
        "hour_sin","hour_cos","dow_sin","dow_cos"
    ] +
    hist_features
)

print("="*60)
print("FEATURE INVENTORY")
print("="*60)
print(f"Total features: {len(FEATURES)}")
print(FEATURES)

print("\nFeature missingness (%):")
print(pd.DataFrame({
    "train_%": train[FEATURES].isna().mean()*100,
    "val_%": val[FEATURES].isna().mean()*100,
    "test_%": test[FEATURES].isna().mean()*100
}).round(2))

FEATURE INVENTORY
Total features: 34
['latitude', 'longitude', 'era5_t2m_c', 'era5_d2m_c', 'era5_sp_hpa', 'era5_tp_mm', 'era5_wind_speed', 'era5_wind_dir', 'era5_blh', 'hour', 'month', 'dayofweek', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'co_lag1', 'co_lag24', 'co_roll24_mean', 'no2_lag1', 'no2_lag24', 'no2_roll24_mean', 'o3_lag1', 'o3_lag24', 'o3_roll24_mean', 'pm10_lag1', 'pm10_lag24', 'pm10_roll24_mean', 'pm25_lag1', 'pm25_lag24', 'pm25_roll24_mean', 'so2_lag1', 'so2_lag24', 'so2_roll24_mean']

Feature missingness (%):
                  train_%  val_%  test_%
latitude             0.00   0.00    0.00
longitude            0.00   0.00    0.00
era5_t2m_c           0.00   0.00    0.00
era5_d2m_c           0.00   0.00    0.00
era5_sp_hpa          0.00   0.00    0.00
era5_tp_mm           0.00   0.00    0.00
era5_wind_speed      0.00   0.00    0.00
era5_wind_dir        0.00   0.00    0.00
era5_blh             0.00   0.00    0.00
hour                 0.00   0.00    0.00
month          

In [8]:
# ============================================================
# CELL 8 — Target Coverage by Split
# ============================================================

print("="*60)
print("TARGET COVERAGE BY SPLIT")
print("="*60)

for name, part in [
    ("Train", train),
    ("Validation", val),
    ("Test", test)
]:
    out = pd.DataFrame({
        "available": part[TARGETS].notna().sum(),
        "missing": part[TARGETS].isna().sum(),
        "missing_%": (part[TARGETS].isna().mean()*100).round(2)
    })

    print(f"\n{name}")
    print(out)

TARGET COVERAGE BY SPLIT

Train
      available  missing  missing_%
co       990602   144373      12.72
no2     1031224   103751       9.14
o3      1025081   109894       9.68
pm10    1001915   133060      11.72
pm25    1029344   105631       9.31
so2      903666   231309      20.38

Validation
      available  missing  missing_%
co       222325    20876       8.58
no2      221133    22068       9.07
o3       219169    24032       9.88
pm10     215755    27446      11.29
pm25     217025    26176      10.76
so2      191870    51331      21.11

Test
      available  missing  missing_%
co       191949    51289      21.09
no2      193169    50069      20.58
o3       192122    51116      21.01
pm10     191569    51669      21.24
pm25     191195    52043      21.40
so2      170003    73235      30.11


In [9]:
# ============================================================
# CELL 9 — Save Preprocessed Splits
# ============================================================

OUT = ROOT / "data/processed"
OUT.mkdir(parents=True, exist_ok=True)

train.to_csv(OUT / "train.csv", index=False)
val.to_csv(OUT / "val.csv", index=False)
test.to_csv(OUT / "test.csv", index=False)

print("="*60)
print("PREPROCESSED SPLITS SAVED")
print("="*60)

print("Train:", OUT / "train.csv")
print("Val  :", OUT / "val.csv")
print("Test :", OUT / "test.csv")

PREPROCESSED SPLITS SAVED
Train: c:\Users\udish\College\MajorProjectSem7\Implementation\data\processed\train.csv
Val  : c:\Users\udish\College\MajorProjectSem7\Implementation\data\processed\val.csv
Test : c:\Users\udish\College\MajorProjectSem7\Implementation\data\processed\test.csv


In [ ]:
# ============================================================
# CELL 10 — Final Audit
# ============================================================
print("="*60)
print("FINAL AUDIT")
print("="*60)
print("Train:", train.shape)
print("Val  :", val.shape)
print("Test :", test.shape)
print("\nMissing total:")
print({
    "train": int(train.isna().sum().sum()),
    "val": int(val.isna().sum().sum()),
    "test": int(test.isna().sum().sum())
})
print("\nDuplicate station-times:")
print({
    "train": int(
        train.duplicated(
            ["location_id","datetime_utc"]).sum()
    ),
    "val": int(
        val.duplicated(
            ["location_id","datetime_utc"]).sum()
    ),
    "test": int(
        test.duplicated(
            ["location_id","datetime_utc"]).sum()
    )
})

FINAL AUDIT
Train: (1134975, 46)
Val  : (243201, 46)
Test : (243238, 46)

Missing total:
{'train': 2959611, 'val': 619327, 'test': 1262774}

Duplicate station-times:
{'train': 0, 'val': 0, 'test': 0}
